## Silver to Gold
Builds the dimensional model from Silver tables.

In [0]:
catalog_name  = dbutils.widgets.get("catalog_name")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema   = dbutils.widgets.get("gold_schema")

print(f"catalog_name       : {catalog_name}")
print(f"silver_schema_name : {silver_schema}")
print(f"gold_schema_name   : {gold_schema}")

catalog_name       : workspace
silver_schema_name : final_project_silver
gold_schema_name   : final_project_gold


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
chicago_df = spark.table(f"{catalog_name}.{silver_schema}.chicago_food_inspection")
dallas_df = spark.table(f"{catalog_name}.{silver_schema}.dallas_food_inspection")

### Load Dimension Tables

In [0]:
# dim_inspection
inspection_cols = [
    "inspection_id",
    "inspection_result",
    "inspection_type",
    "risk",
    "source_table",
    "load_date"
]

inspection_chicago_clean = (
    chicago_df
    .select(*inspection_cols)
    .dropDuplicates(["inspection_id"])
)

inspection_dallas_clean = (
    dallas_df
    .select(*inspection_cols)
    .dropDuplicates(["inspection_id"])
)

dim_inspection = (
    inspection_chicago_clean
    .unionByName(inspection_dallas_clean)
    .withColumn("inspection_key", F.monotonically_increasing_id())
)

dim_inspection.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_inspection")

print("dim_inspection written")

dim_inspection written


In [0]:
# dim_violation
violation_cols = [
    "violation_code",
    "violation_description",
    "source_table",
    "load_date"
]

violation_chicago_clean = (
    chicago_df
    .select(*violation_cols)
    .dropDuplicates(["violation_code"])
)

violation_dallas_clean = (
    dallas_df
    .select(*violation_cols)
    .dropDuplicates(["violation_code"])
)

dim_violation = (
    violation_chicago_clean
    .unionByName(violation_dallas_clean)
    .withColumn("violation_key", F.monotonically_increasing_id())
)

dim_violation.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_violation")

print("dim_violation written")

dim_violation written


In [0]:
# dim_date
dim_date = (
    chicago_df.select("inspection_date")
    .unionByName(dallas_df.select("inspection_date"))
)

dim_date = (
    dim_date
    .select(F.to_date(F.col("inspection_date")).alias("inspection_date"))
    .dropDuplicates()
    .withColumn("date_key",   F.date_format(F.col("inspection_date"), "yyyyMMdd").cast("int"))
    .withColumn("year",         F.year(F.col("inspection_date")))
    .withColumn("quarter",      F.concat(F.lit("Q"), F.quarter(F.col("inspection_date"))))
    .withColumn("month_of_year",F.month(F.col("inspection_date")))
    .withColumn("week_of_year", F.weekofyear(F.col("inspection_date")))
    .withColumn("day_of_year",  F.dayofyear(F.col("inspection_date")))
    .withColumn("month_name",   F.date_format(F.col("inspection_date"), "MMMM"))
    .withColumn("day_of_month", F.dayofmonth(F.col("inspection_date")))
    .withColumn("day_of_week",  F.dayofweek(F.col("inspection_date")))
    .withColumn("day_name",     F.date_format(F.col("inspection_date"), "EEEE"))
    .withColumn("is_weekend",   F.dayofweek(F.col("inspection_date")).isin([1, 7]).cast("boolean"))
)

if not spark.catalog.tableExists(f"{silver_schema}.dim_date"):
    dim_date.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_date")
    print("first load completed")
else:
    DeltaTable.forName(spark, f"{gold_schema}.dim_date").alias("target") \
        .merge(dim_date.alias("source"), "target.date_key = source.date_key") \
        .whenNotMatchedInsertAll() \
        .execute()
    print("incremental load completed")

first load completed


In [0]:
# dim_comment
bridge_cols = [
    "inspection_id",
    "violation_code",
    "violation_comments"
]

bridge_chicago = (chicago_df.select(*bridge_cols))

bridge_chicago = (
    bridge_chicago
    .join(dim_inspection.select("inspection_id", "inspection_key"),
          on="inspection_id",
          how="inner")
    .join(dim_violation.select("violation_code", "violation_key"),
          on="violation_code",
          how="inner")
)

bridge_chicago = bridge_chicago.select("inspection_key", "violation_key", "violation_comments")

bridge_chicago = bridge_chicago.dropDuplicates(["inspection_key", "violation_key"])

bridge_dallas = (dallas_df.select(*bridge_cols))

bridge_dallas = (
    bridge_dallas
    .join(dim_inspection.select("inspection_id", "inspection_key"),
          on="inspection_id",
          how="inner")
    .join(dim_violation.select("violation_code", "violation_key"),
          on="violation_code",
          how="inner")
)

bridge_dallas = bridge_dallas.select("inspection_key", "violation_key", "violation_comments")

bridge_dallas = bridge_dallas.dropDuplicates(["inspection_key", "violation_key"])

dim_comment = (
    bridge_chicago
    .unionByName(bridge_dallas)
    .withColumn("comment_key", F.monotonically_increasing_id())
)

dim_comment.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_comment")

print("dim_comment written")

dim_comment written


In [0]:
# dim_establishment
establishment_cols = [
    "establishment_id",
    "establishment_name",
    "aka_establishment_name",
    "license_number",
    "facility_type",
    "street_address",
    "city",
    "zip_code",
    "latitude",
    "longitude",
    "location",
    "source_table",
    "load_date"
]

establishment_chicago_clean = (
    chicago_df
    .select(*establishment_cols)
    .dropDuplicates(['establishment_id', 'license_number'])
)

establishment_dallas_clean = (
    dallas_df
    .select(*establishment_cols)
    .dropDuplicates(['establishment_id', 'license_number'])
)

establishment_union = (
    establishment_chicago_clean
    .unionByName(establishment_dallas_clean)
)

In [0]:
# dim_establishment
# Load dimension
if spark.catalog.tableExists(f"{gold_schema}.dim_establishment"):
    dim_df = spark.table(f"{gold_schema}.dim_establishment")
else:
    dim_df = None


# Create table if not exists
if dim_df is None:
    window_init = Window.orderBy("establishment_id", "license_number")

    empty_df = establishment_union \
        .withColumn("establishment_key", F.row_number().over(window_init)) \
        .withColumn("start_date", F.current_timestamp()) \
        .withColumn("end_date", F.lit(None).cast("timestamp")) \
        .withColumn("is_current", F.lit(True))

    empty_df.write.format("delta").mode("overwrite").saveAsTable(f"{gold_schema}.dim_establishment")

    dim_df = spark.table(f"{gold_schema}.dim_establishment")


# Identify current records
current_dim = dim_df.filter("is_current = true")


# Identify new or changed rows
new_or_changed = (
    establishment_union.alias("s")
    .join(
        current_dim.alias("t"),
        (F.col("s.establishment_id") == F.col("t.establishment_id")) &
        (F.col("s.license_number") == F.col("t.license_number")),
        "left"
    )
    .filter("""
        t.establishment_id IS NULL
        OR t.facility_type <> s.facility_type
    """)
)


# Close old records (SCD2 update)
dim_table = DeltaTable.forName(spark, f"{gold_schema}.dim_establishment")

dim_table.alias("t").merge(
    establishment_union.alias("s"),
    "t.establishment_id = s.establishment_id AND t.license_number = s.license_number AND t.is_current = true"
).whenMatchedUpdate(
    condition="t.facility_type <> s.facility_type",
    set={
        "end_date": F.current_timestamp(),
        "is_current": F.lit(False)
    }
).execute()


# Generate surrogate keys (max + row_number)
# Re-read AFTER the merge so max_key reflects any newly closed records
dim_df = spark.table(f"{gold_schema}.dim_establishment")
max_key = dim_df.agg(F.max("establishment_key")).collect()[0][0]
if max_key is None:
    max_key = 0

window_new = Window.orderBy("s.establishment_id", "s.license_number")

new_rows_with_key = (
    new_or_changed
    .withColumn("rn", F.row_number().over(window_new))
    .withColumn("establishment_key", F.lit(max_key) + F.col("rn"))
)


# Prepare insert rows
final_inserts = new_rows_with_key.select(
    "establishment_key",
    "s.establishment_id",
    "s.establishment_name",
    "s.aka_establishment_name",
    "s.license_number",
    "s.facility_type",
    "s.street_address",
    "s.city",
    "s.zip_code",
    "s.latitude",
    "s.longitude",
    "s.location",
    "s.source_table",
    "s.load_date",
    F.current_timestamp().alias("start_date"),
    F.lit(None).cast("timestamp").alias("end_date"),
    F.lit(True).alias("is_current")
)


# Insert into dimension
final_inserts.write.format("delta").mode("append").saveAsTable(f"{gold_schema}.dim_establishment")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


### Load Fact Table

In [0]:
# fact_food_inspection
inspection_df    = spark.table(f"{gold_schema}.dim_inspection")
date_df          = spark.table(f"{gold_schema}.dim_date")
establishment_df = spark.table(f"{gold_schema}.dim_establishment").filter("is_current = true")

fact_cols = [
    "inspection_id",
    "establishment_id",
    "license_number",
    "inspection_date",
    "inspection_score"
]

fact_chicago = (chicago_df.select(*fact_cols).dropDuplicates(["inspection_id"]))
fact_dallas  = (dallas_df.select(*fact_cols).dropDuplicates(["inspection_id"]))

fact_union = (fact_chicago.unionByName(fact_dallas))

# Incremental load — filter out records that already exist in the fact table
if spark.catalog.tableExists(f"{gold_schema}.fact_food_inspection"):
    existing_df = spark.table(f"{gold_schema}.fact_food_inspection")
    max_key = existing_df.agg(F.max("food_inspection_key")).collect()[0][0]
    if max_key is None:
        max_key = 0

    # Get existing inspection_keys already loaded
    existing_keys = existing_df.select("inspection_key").distinct()

    # Only keep new records not yet in the fact table
    fact_union = (
        fact_union
        .join(inspection_df.select("inspection_id", "inspection_key"),
              on="inspection_id",
              how="inner")
        .join(existing_keys,
              on="inspection_key",
              how="left_anti")  # exclude already loaded records
    )
else:
    max_key = 0
    fact_union = (
        fact_union
        .join(inspection_df.select("inspection_id", "inspection_key"),
              on="inspection_id",
              how="inner")
    )

# Join remaining dimensions
fact_food_inspection = (
    fact_union
    .join(date_df.select("inspection_date", "date_key"),
          on="inspection_date",
          how="inner")
    .join(establishment_df.select("establishment_id", "license_number", "establishment_key"),
          on=["establishment_id", "license_number"],
          how="inner")
)

# Generate surrogate key
window_fact = Window.orderBy("inspection_key", "establishment_key")

fact_food_inspection = (
    fact_food_inspection
    .withColumn("rn", F.row_number().over(window_fact))
    .withColumn("food_inspection_key", F.lit(max_key) + F.col("rn"))
    .drop("rn")
)

fact_food_inspection = fact_food_inspection.select(
    "food_inspection_key",
    "inspection_key",
    "establishment_key",
    "date_key",
    "inspection_score"
)

fact_food_inspection.write.format("delta").mode("append").saveAsTable(f"{gold_schema}.fact_food_inspection")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
